In [1]:
from dotenv import load_dotenv
import os
import requests
import pandas as pd
import time
import calendar

In [2]:
# Load API key
load_dotenv("../../.env")
api_key = os.getenv("NOAA_API_KEY")

headers = {"token": api_key}

# NOAA API endpoint
BASE_URL = "https://www.ncdc.noaa.gov/cdo-web/api/v2"

# Years to download
START_YEAR = 2015
END_YEAR = 2020

# Maximum stations to collect (increase later if needed)
MAX_STATIONS = 50

all_weather = []

In [3]:
FIPS_TO_STATE = {
    "01":"AL","02":"AK","04":"AZ","05":"AR","06":"CA","08":"CO","09":"CT","10":"DE","11":"DC","12":"FL",
    "13":"GA","15":"HI","16":"ID","17":"IL","18":"IN","19":"IA","20":"KS","21":"KY","22":"LA","23":"ME",
    "24":"MD","25":"MA","26":"MI","27":"MN","28":"MS","29":"MO","30":"MT","31":"NE","32":"NV","33":"NH",
    "34":"NJ","35":"NM","36":"NY","37":"NC","38":"ND","39":"OH","40":"OK","41":"OR","42":"PA","44":"RI",
    "45":"SC","46":"SD","47":"TN","48":"TX","49":"UT","50":"VT","51":"VA","53":"WA","54":"WV","55":"WI","56":"WY"
}

In [4]:
def get_json_with_retries(url, headers, params, timeout=(10, 90), max_tries=6):
    """
    timeout=(connect_timeout, read_timeout)
    Retries on timeouts, 429, and 5xx errors with exponential backoff.
    """
    for attempt in range(1, max_tries + 1):
        try:
            r = requests.get(url, headers=headers, params=params, timeout=timeout)

            # Rate limit
            if r.status_code == 429:
                wait = min(60, 2 ** attempt)
                print(f"429 rate limit. Waiting {wait}s...")
                time.sleep(wait)
                continue

            # Server hiccups
            if 500 <= r.status_code < 600:
                wait = min(60, 2 ** attempt)
                print(f"{r.status_code} server error. Waiting {wait}s...")
                time.sleep(wait)
                continue

            r.raise_for_status()
            return r.json()

        except (requests.exceptions.ReadTimeout, requests.exceptions.ConnectTimeout) as e:
            wait = min(60, 2 ** attempt)
            print(f"Timeout (attempt {attempt}/{max_tries}). Waiting {wait}s...")
            time.sleep(wait)

        except requests.exceptions.RequestException as e:
            # For other request errors, retry a bit then raise
            wait = min(60, 2 ** attempt)
            print(f"Request error: {e}. Waiting {wait}s...")
            time.sleep(wait)

    raise RuntimeError(f"Failed after {max_tries} tries: {url} params={params}")

In [5]:
def fetch_all_stations_for_state_fips(fips2: str, limit: int = 1000, sleep_s: float = 0.15):
    """Fetch all GHCND stations for a given 2-digit state FIPS code."""
    locationid = f"FIPS:{fips2}"
    stations_url = f"{BASE_URL}/stations"

    all_rows = []
    offset = 1

    while True:
        params = {
            "datasetid": "GHCND",
            "locationid": locationid,
            "limit": limit,
            "offset": offset
        }
        r = requests.get(stations_url, headers=headers, params=params, timeout=30)

        # If NOAA ever rate-limits you, this helps a bit
        if r.status_code == 429:
            time.sleep(2.0)
            continue

        r.raise_for_status()
        d = r.json()

        batch = d.get("results", [])
        all_rows.extend(batch)

        meta = d.get("metadata", {}).get("resultset", {})
        count = meta.get("count", 0)

        if offset + limit > count:
            break

        offset += limit
        time.sleep(sleep_s)

    return all_rows

In [24]:
MAX_PER_STATE = 300
all_stations = []
done_states = set()

# If you previously saved partial progress, load it here:
# stations_partial = pd.read_csv("stations_partial.csv")
# done_states = set(stations_partial["state"].unique())
# all_stations = [stations_partial]

for fips2, st in FIPS_TO_STATE.items():
    if st in done_states:
        print(f"{st}: already done, skipping")
        continue

    rows = fetch_all_stations_for_state_fips(fips2)
    df = pd.DataFrame(rows)

    if df.empty:
        print(f"{st} ({fips2}): 0 stations")
        done_states.add(st)
        continue

    # extra safety: keep US stations
    df = df[df["id"].str.contains(":US", na=False)].copy()
    df["state"] = st

    # sample up to 500
    if len(df) > MAX_PER_STATE:
        df = df.sample(MAX_PER_STATE, random_state=42)

    all_stations.append(df)
    done_states.add(st)

    stations_partial = pd.concat(all_stations, ignore_index=True)
    stations_partial.to_csv("stations_partial.csv", index=False)  # <- progress checkpoint
    print(f"{st} ({fips2}): kept {len(df)} stations (saved progress)")

AL (01): kept 300 stations (saved progress)
AK (02): kept 300 stations (saved progress)
AZ (04): kept 300 stations (saved progress)
AR (05): kept 300 stations (saved progress)
CA (06): kept 300 stations (saved progress)
CO (08): kept 300 stations (saved progress)
CT (09): kept 300 stations (saved progress)
DE (10): kept 148 stations (saved progress)
DC (11): kept 23 stations (saved progress)
FL (12): kept 300 stations (saved progress)
GA (13): kept 300 stations (saved progress)
HI (15): kept 300 stations (saved progress)
ID (16): kept 300 stations (saved progress)
IL (17): kept 300 stations (saved progress)
IN (18): kept 300 stations (saved progress)
IA (19): kept 300 stations (saved progress)
KS (20): kept 300 stations (saved progress)
KY (21): kept 300 stations (saved progress)
LA (22): kept 300 stations (saved progress)
ME (23): kept 300 stations (saved progress)
MD (24): kept 300 stations (saved progress)
MA (25): kept 300 stations (saved progress)
MI (26): kept 300 stations (saved

In [6]:
MAX_PER_STATE = 10  # change to 50/100/200 depending on speed you want

stations_df = pd.read_csv("stations.csv")

# optional: prefer higher data coverage and more recent maxdate
stations_df["maxdate"] = pd.to_datetime(stations_df["maxdate"], errors="coerce")

stations_small = (
    stations_df.sort_values(["state", "datacoverage", "maxdate"], ascending=[True, False, False])
              .groupby("state", group_keys=False)
              .head(MAX_PER_STATE)
              .reset_index(drop=True)
)

stations_small.to_csv(f"stations_sampled_{MAX_PER_STATE}_per_state.csv", index=False)
print("Total stations:", len(stations_small))
print(stations_small["state"].value_counts().head())

Total stations: 510
AK    10
PA    10
ND    10
NE    10
NH    10
Name: state, dtype: int64


In [6]:
def get_json_with_retries(url, headers, params, timeout=(10, 90), max_tries=6):
    for attempt in range(1, max_tries + 1):
        try:
            r = requests.get(url, headers=headers, params=params, timeout=timeout)

            if r.status_code == 429:
                time.sleep(min(60, 2 ** attempt))
                continue
            if 500 <= r.status_code < 600:
                time.sleep(min(60, 2 ** attempt))
                continue

            # show helpful text for 400s if they happen
            if r.status_code != 200:
                print("HTTP", r.status_code, "URL:", r.url)
                print("BODY:", r.text[:300])

            r.raise_for_status()
            return r.json()

        except (requests.exceptions.ReadTimeout, requests.exceptions.ConnectTimeout):
            time.sleep(min(60, 2 ** attempt))

    raise RuntimeError(f"Failed after {max_tries} tries: {url} params={params}")


def fetch_tmax_station_batch_month(station_ids_batch, year, month, limit=1000, sleep_s=0.1):
    url = f"{BASE_URL}/data"
    last_day = calendar.monthrange(year, month)[1]
    startdate = f"{year}-{month:02d}-01"
    enddate   = f"{year}-{month:02d}-{last_day:02d}"

    out = []
    offset = 1

    while True:
        params = {
            "datasetid": "GHCND",
            "stationid": station_ids_batch,   # <-- batch list
            "startdate": startdate,
            "enddate": enddate,
            "datatypeid": "TMAX",
            "units": "standard",
            "limit": limit,
            "offset": offset
        }
        d = get_json_with_retries(url, headers=headers, params=params, timeout=(10, 90))
        batch = d.get("results", [])
        out.extend(batch)

        meta = d.get("metadata", {}).get("resultset", {})
        count = meta.get("count", 0)
        if offset + limit > count:
            break
        offset += limit
        time.sleep(sleep_s)

    return out

In [ ]:
stations_small = pd.read_csv("stations_sampled_10_per_state.csv")
all_station_ids = stations_small["id"].unique().tolist()

station_to_state = dict(zip(stations_small["id"], stations_small["state"]))

BATCH_SIZE = 20
years = range(2020, 2025)

chunks = []
out_path = "daily_tmax_partial.csv"

for y in years:
    for m in range(1, 13):
        print(f"{y}-{m:02d}")
        for i in range(0, len(all_station_ids), BATCH_SIZE):
            batch_ids = all_station_ids[i:i+BATCH_SIZE]
            rows = fetch_tmax_station_batch_month(batch_ids, y, m)
            if not rows:
                continue
            df = pd.DataFrame(rows)

            df["state"] = df["station"].map(station_to_state)

            chunks.append(df)

        # checkpoint after each month
        partial = pd.concat(chunks, ignore_index=True)


        partial.to_csv(out_path, index=False)
        print(" saved rows:", len(partial))

2020-01
 saved rows: 3524
2020-02
 saved rows: 6803
2020-03
 saved rows: 10303
2020-04
 saved rows: 13687
2020-05
 saved rows: 17176
2020-06
 saved rows: 20539
2020-07
 saved rows: 24031
2020-08
 saved rows: 27479
2020-09
 saved rows: 30829
2020-10
 saved rows: 34286
2020-11
 saved rows: 37612
2020-12
 saved rows: 41018
2021-01
 saved rows: 44445
2021-02
 saved rows: 47572
2021-03
 saved rows: 51000
2021-04
 saved rows: 54327
2021-05
 saved rows: 57756
2021-06
 saved rows: 61073
2021-07
 saved rows: 64492
2021-08
 saved rows: 67925
2021-09
 saved rows: 71234
2021-10
 saved rows: 74656
2021-11
 saved rows: 77980
2021-12
 saved rows: 81390
2022-01
 saved rows: 84811
2022-02
 saved rows: 87910
2022-03
 saved rows: 91322
2022-04
 saved rows: 94641
2022-05
 saved rows: 98060
2022-06
 saved rows: 101362
2022-07
 saved rows: 104792
2022-08
 saved rows: 108228
2022-09
 saved rows: 111529
2022-10
 saved rows: 114964
2022-11
 saved rows: 118279
2022-12
 saved rows: 121705
2023-01
 saved rows: 12